# IMPORT MODULES, INSTANTIATE CLASS

In [1]:
import glob
import os

import pandas as pd

from geoai.utils_geo.RasterOps import RasterOperations
from geoai.utils_ds.DataFrameOps import DataFrameOperations
from geoai.utils_geo.VectorOps import VectorOperations

raster_ops = RasterOperations()
df_ops = DataFrameOperations()
vector_ops = VectorOperations()

# SET GLOBAL VAR

In [2]:
RASTER_PATH = r"raster_files\QC_2018_2023.tif"
N_BANDS = 5
BAND_NAMES = ["BLUE", "GREEN", "RED", "NIR", "SWIR"]

# CLIP ROI SHAPEFILE TO RASTER

In [3]:
for shapefile in glob.glob(os.path.join("shapefiles", "*.shp")):
    roi_name = os.path.splitext(os.path.basename(shapefile))[0]
    output_raster_path = os.path.join("raster_files", f"{roi_name}.tif")
    vector_ops.clip_raster_with_shapefile(RASTER_PATH, shapefile, output_raster_path)
    print(f"Done clipping {roi_name}")

Done clipping builtup
Done clipping grass
Done clipping road
Done clipping trees


# MAKE A DF ON EVERY ROI RASTER

In [4]:
list_of_roi = ["builtup", "trees", "grass", "road"]
for roi_path in glob.glob(os.path.join("raster_files", "*.tif")):
    roi_name = os.path.splitext(os.path.basename(roi_path))[0]
    if roi_name in list_of_roi:  
        df_bands = []
        array = raster_ops.raster_to_array(roi_path)
        for band_index, band_name in zip(range(N_BANDS), BAND_NAMES):
            flat = raster_ops.flatten_array(array, band_index)
            df = df_ops.convert_to_df(flat, band_name)
            df = df.loc[~(df==0).all(axis=1)] # remove rows if all of its column is zero
            df_bands.append(df)
        final_df_per_bands = pd.concat(df_bands, axis=1) 
        final_df_per_bands["Landcover"] = roi_name
        final_df_per_bands.to_csv(f"csv_files\{roi_name}.csv", index = False)
        print(final_df_per_bands)

Converting raster array to DataFrame with column name BLUE
Converting raster array to DataFrame with column name GREEN
Converting raster array to DataFrame with column name RED
Converting raster array to DataFrame with column name NIR
Converting raster array to DataFrame with column name SWIR
               BLUE        GREEN          RED          NIR         SWIR  \
483      835.333313   920.750000  1301.000000  1478.666626  1976.750000   
484      938.666687   966.333313  1153.000000  1376.000000  2025.199951   
485      871.000000   902.000000   946.000000  1218.000000  2025.199951   
486      985.000000  1050.000000  1114.666626  1573.000000  1974.750000   
963      940.250000   994.500000  1046.000000  1256.500000  1976.750000   
...             ...          ...          ...          ...          ...   
130084  1017.500000  1097.500000  1321.333374  1544.000000  2278.000000   
130085  1017.500000  1097.500000  1321.333374  1544.000000  2278.000000   
130086   968.333313   983.66668

# CREATE THE TRAINING DF THAT CONSIST OF ALL THE DATA FROM ROIs

In [5]:
df_roi = []
for roi_csv_path in glob.glob(os.path.join("csv_files", "*.csv")):
    roi_csv_name = os.path.splitext(os.path.basename(roi_csv_path))[0]
    if roi_csv_name in list_of_roi:
        df = pd.read_csv(roi_csv_path)
        df_roi.append(df)
final_training_data = pd.concat(df_roi, axis=0)
final_training_data.to_csv("csv_files/dataset.csv", index=False)

END